# 01 · From DuckDB marts to a modeling dataset

        **Question:** Can county characteristics explain differences in home values across the United States?

        This is a reconstructed walkthrough of the full study completed October 6, 2026.
        It documents the final analysis; it is not a contemporaneous exploration log.
        The target is ACS median owner-occupied home value in **constant 2024 dollars**,
        not a sale price or an individual-home appraisal. Associations are not causal effects.

        Run all cells for the packaged evidence walkthrough. Original DuckDB marts are
        required only for the optional rebuild. See [README](README.md) for setup and provenance.
        Next: [training and validation](02_training_and_validation.ipynb).

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Works from the notebook directory or the source/bundle root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/modeling/county_characteristics.py").exists())
sys.path.insert(0, str(ROOT))
NB_DIR = ROOT / "notebooks/03_modeling"
EVIDENCE = NB_DIR / "evidence"
def read_csv(name):
    return pd.read_csv(EVIDENCE / name, dtype={"county_fips": str})
def read_json(name):
    return json.loads((EVIDENCE / name).read_text(encoding="utf-8"))

# Fail loudly if a packaged artifact has changed since publication.
provenance = read_json("snapshot.json")
for name, digest in provenance["sha256"].items():
    assert hashlib.sha256((EVIDENCE / name).read_bytes()).hexdigest() == digest, name
panel = pd.read_parquet(EVIDENCE / "county_period_panel.parquet")
manifest = read_csv("feature_manifest.csv")
groups = {group: data.feature.tolist() for group, data in
          manifest.loc[manifest.role.eq("primary")].groupby("group")}
features = [feature for columns in groups.values() for feature in columns]
from src.modeling.county_price import TARGET, regression_metrics
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 90)
print("Evidence: completed full study, 2026-10-06; equal county-period weights.")

Evidence: completed full study, 2026-10-06; equal county-period weights.


## 1. Define the observation and the sample

        One row represents one county and one five-year ACS period. We use the 50 states
        and DC and give each county equal weight within a period. The three windows do
        not overlap: 2010–2014, 2015–2019, and 2020–2024. Adjacent annual ACS releases
        share four survey years ([Census guidance](https://www.census.gov/programs-surveys/acs/guidance/comparing-acs-data.html)).
        Cross sections retain eligible counties; the longitudinal analysis applies a
        stricter geography screen. County identifiers are strings so leading zeros survive.

In [2]:
from src.modeling.county_characteristics import assert_unique, STATE_CODES
assert_unique(panel, ["county_fips", "year"])
assert panel.county_fips.str.fullmatch(r"\d{5}").all()
assert set(panel.year) == {2014, 2019, 2024}
assert panel.county_fips.str[:2].isin(STATE_CODES).all()
assert panel[TARGET].gt(0).all()
np.testing.assert_allclose(panel.log_value, np.log(panel[TARGET]))
display(panel.groupby(["period_start", "year"]).agg(
    counties=("county_fips", "nunique"), median_value=(TARGET, "median")))
print(f"{len(panel):,} unique county-period observations")

,,counties,median_value
period_start,year,,
2010,2014,3141,145491.400547
2015,2019,3140,156932.229902
2020,2024,3139,185300.000000


9,420 unique county-period observations


## 2. Audit the joins before modeling

        The production loader joins housing, economic and social marts, then attaches
        hazards, climate and county labels. Duplicate keys fail the run; rows lost in the
        inner join are recorded. Here we inspect the full-run audit, not just the small example.
        Missing metro labels remain `Unknown`. Exclusion counts below are source county-year
        rows by reason, not necessarily distinct study counties.

In [3]:
audit = read_csv("source_key_audit.csv")
assert audit.duplicate_keys.eq(0).all()
display(audit)
exclusions = read_csv("exclusions.csv")
display(exclusions.groupby("reason").size().rename("source_county_year_rows").to_frame())
geography = read_csv("geography_audit.csv")
display(geography.groupby(["boundary_review", "balanced_geography"]).size().rename("counties").to_frame())

,table,duplicate_keys
0,county_housing,0
1,county_economic_annual,0
2,county_social_annual,0
3,county_fema_annual,0
4,county_noaa_storm_annual,0
5,county_climate_annual,0


,source_county_year_rows
reason,
Missing or nonpositive target,8
Outside 50 states and DC,1170


counties
boundary_review balanced_geography          
False           False                      4
                True                    3122
True            False                     28

## 3. Establish a feature contract

        Economic, social, housing structure and hazard variables form the primary design.
        Climate is a covered-county sensitivity analysis. Affordability and mortgage measures
        are separate because they can already reflect housing costs. Previous values,
        geographic price summaries, IDs and target-derived ratios are excluded.

        Definitions must exist in all three periods. Availability is then checked using
        **2014 only**: broadband/computer measures have no initial observed values.
        Redundant owner/renter and mortgage/no-mortgage categories are not both included.

In [4]:
display(manifest.groupby(["role", "group"]).size().rename("features").to_frame())
display(manifest.loc[manifest.role.ne("primary"), ["source_feature", "role", "reason"]])
assert len(features) == 57
assert TARGET not in features
assert not set(features) & {"county_fips", "state", "msa_code", "log_value"}
assert not any("price" in f or "home_value" in f for f in features)
print("Primary model inputs:", len(features))

features
role        group                  
excluded    affordability         1
            economic              1
            housing               2
            social                6
primary     economic             27
            hazards               5
            housing              12
            social               13
sensitivity affordability         5
            climate               4
target      housing               1

,source_feature,role,reason
9,cohabiting_couple_household_pct,excluded,Definition unavailable in one or more study periods
16,gross_rent_30_to_34_9_pct,sensitivity,May reflect housing costs; evaluated separately
17,gross_rent_35_plus_pct,sensitivity,May reflect housing costs; evaluated separately
20,householder_moved_1989_or_earlier_pct,excluded,Definition unavailable in one or more study periods
23,households_without_broadband_pct,excluded,No observed values in initial 2014 training period
24,households_without_computer_pct,excluded,No observed values in initial 2014 training period
31,married_couple_household_pct,excluded,Definition unavailable in one or more study periods
45,owner_cost_30_to_34_9_pct,sensitivity,May reflect housing costs; evaluated separately
46,owner_cost_35_plus_pct,sensitivity,May reflect housing costs; evaluated separately
52,renter_occupied_pct,excluded,Complement of another included percentage


Primary model inputs: 57


## 4. Engineer features with the same code used in production

        Negative ACS sentinels and impossible percentages become missing. Percentages stay
        in percentage points; dollar and count predictors use `log1p`. The target uses `log`.
        Hazard counts/damage sum over the matching five years, requiring five observed years.
        Climate uses five-year means, also requiring five observations. Valid negative
        temperatures remain valid. Event marts already treat absent records as zero.

        The real annual sample below contains Autauga County, Los Angeles County and Denver
        County. It demonstrates the transformation; all headline results use the full panel.
        The function source is displayed to make the transformations reviewable.

In [5]:
import inspect
from src.modeling.county_characteristics import (
    feature_manifest, build_study_panel, screen_training_availability)
print(inspect.getsource(build_study_panel))

def build_study_panel(annual: pd.DataFrame, manifest: pd.DataFrame):
    assert_unique(annual, ['county_fips', 'year'])
    annual = annual.sort_values(['county_fips', 'year']).copy()
    annual['state'] = annual.county_fips.str[:2].map(STATE_CODES)
    excluded = annual.loc[annual.state.isna(), ['county_fips', 'county_name', 'year']].copy()
    excluded['reason'] = 'Outside 50 states and DC'
    annual = annual.loc[annual.state.notna()].copy()
    numeric = annual.select_dtypes(include='number').columns
    annual[numeric] = annual[numeric].replace([np.inf, -np.inf], np.nan)
    # Census special negative estimates and impossible percentages cannot enter a fit.
    for row in manifest.itertuples():
        col = row.source_feature
        if col in annual and col not in CLIMATE_COLUMNS:
            annual.loc[annual[col].lt(0), col] = np.nan
            if col.endswith('_pct'):
                annual.loc[annual[col].gt(100), col] = np.nan
    panel = annual.loc[annual.year.isin(PERIODS

In [6]:
registry = read_csv("feature_definitions_by_year.csv")
annual_example = pd.read_parquet(EVIDENCE / "annual_example.parquet")
clean_example, engineered, _ = build_study_panel(annual_example, feature_manifest(registry))
income = "median_household_income_2024_usd"
display(engineered[["county_fips", "year", income, "log1p__" + income,
                    "storm_event_count", "log1p__period_sum__storm_event_count"]])
np.testing.assert_allclose(engineered["log1p__" + income], np.log1p(engineered[income]))
window = clean_example.loc[clean_example.year.between(2010, 2014)]
totals = window.groupby("county_fips").storm_event_count.sum(min_count=5)
first = engineered.loc[engineered.year.eq(2014)]
np.testing.assert_allclose(first.log1p__period_sum__storm_event_count,
                          np.log1p(first.county_fips.map(totals)), equal_nan=True)
original = panel.set_index(["county_fips", "year"])
rebuilt = engineered.set_index(["county_fips", "year"])
np.testing.assert_allclose(rebuilt[features], original.loc[rebuilt.index, features], equal_nan=True)
print("Example transformations match the published panel.")

,county_fips,year,median_household_income_2024_usd,log1p__median_household_income_2024_usd,storm_event_count,log1p__period_sum__storm_event_count
0,01001,2014,69532.433914,11.149563,11.0,4.532599
1,01001,2019,72062.445617,11.185302,7.0,3.583519
2,01001,2024,72481.000000,11.191094,6.0,3.637586
3,06037,2014,74031.006818,11.212253,31.0,4.875197
4,06037,2019,83489.418698,11.332487,14.0,4.624973
5,06037,2024,90112.000000,11.408820,58.0,5.252273
6,08031,2014,68638.019566,11.136616,27.0,4.510860
7,08031,2019,84161.810113,11.340508,35.0,4.859812
8,08031,2024,94718.000000,11.458670,54.0,5.187386


Example transformations match the published panel.


## 5. Check missingness without learning preprocessing from the test set

        Missingness is reported for all periods as a data diagnostic. Median imputation,
        missingness indicators and linear-model standardization are fitted separately
        inside every training fold. Nothing is imputed here.

In [7]:
screened = screen_training_availability(feature_manifest(registry), panel)
assert set(screened.loc[screened.role.eq("primary"), "feature"]) == set(features)
missing = read_csv("missingness.csv").pivot(index="feature", columns="year", values="missing_fraction")
display(missing.loc[missing.max(axis=1).sort_values(ascending=False).head(12).index].round(3))
display(manifest.loc[manifest.reason.str.contains("No observed"), ["source_feature", "reason"]])

year,2014,2019,2024
feature,,,
log1p__mean_cash_public_assistance_income_2024_usd,0.018,0.059,0.054
log1p__mean_supplemental_security_income_2024_usd,0.004,0.012,0.014
log1p__women_with_birth_past_12_months_count,0.008,0.000,0.000
log1p__period_sum__storm_damage_2024_usd,0.000,0.000,0.003
log1p__period_sum__storm_deaths,0.000,0.000,0.003
log1p__period_sum__disaster_count,0.000,0.000,0.003
log1p__period_sum__storm_event_count,0.000,0.000,0.003
log1p__period_sum__storm_injuries,0.000,0.000,0.003
log1p__median_household_income_2024_usd,0.000,0.000,0.000


,source_feature,reason
23,households_without_broadband_pct,No observed values in initial 2014 training period
24,households_without_computer_pct,No observed values in initial 2014 training period


## 6. Optional: rebuild the full panel from DuckDB

        This branch reads the original database without writing to it. It reproduces the
        join, geography-label correction, feature construction and training-availability
        screen. The complete CLI additionally exports the join/exclusion audits and all
        modeling experiments. Set the flag only when the marts are available locally.

In [8]:
REBUILD_FROM_MARTS = False
DATABASE = ROOT / "data/housing_predict.duckdb"
if REBUILD_FROM_MARTS:
    import duckdb
    from src.modeling.county_price import load_county_year_data
    with duckdb.connect(str(DATABASE), read_only=True) as con:
        for table in read_csv("source_key_audit.csv")["table"]:
            assert con.execute(f"SELECT count(*) - count(DISTINCT (county_fips, year)) FROM feature.{table}").fetchone()[0] == 0
        reference = con.execute("SELECT lpad(fips, 5, '0') AS county_fips FROM raw.counties").df()
        assert_unique(reference, ["county_fips"])
        full_registry = con.execute("SELECT * FROM feature.canonical_feature_registry").df()
        annual, _, _ = load_county_year_data(con)
    annual.loc[~annual.county_fips.isin(reference.county_fips), "msa_type"] = "Unknown"
    _, rebuilt_panel, _ = build_study_panel(annual, feature_manifest(full_registry))
    rebuilt_manifest = screen_training_availability(feature_manifest(full_registry), rebuilt_panel)
    pd.testing.assert_frame_equal(rebuilt_panel, panel)
    print("Full panel matches the saved evidence.")
else:
    print("Using verified packaged evidence; the original database is not required.")

Using verified packaged evidence; the original database is not required.


## What this establishes

        The full panel contains 9,420 observations and 57 primary inputs. The annual sample
        reproduces the feature transformations. Known substantial county changes are
        flagged rather than resolved with an invented crosswalk; minor boundary changes
        and ACS sampling uncertainty remain limitations.

        [Continue to training and validation →](02_training_and_validation.ipynb)